# Simple CrewAI Ticket Solver

The same customer-support example as the adjacent Python, LCEL, LangGraph and Microsoft notebooks: clean, analyse, decide, reply. Same sample, prompts, five task invocations and result dictionary.

A native CrewAI Flow controls order. A small helper calls the LLM directly; no Crew, Agent or Task is needed. No model-selected tools, conversation-history reuse or real escalation action is added.

## 1. Install
Use Colab CPU or Jupyter with Python 3.10–3.13. Run in order and restart after installation if requested. Enable Notebook access for OPENAI_API_KEY in Colab Secrets. The demo uses paid API calls.


## Google Colab setup

Use a fresh **Python 3.12 or 3.13 CPU runtime** for each notebook. In the Secrets sidebar, add `OPENAI_API_KEY` and enable **Notebook access** for this notebook. Run cells from top to bottom. If installation requests a restart, restart the session and continue with the imports/credentials cell (do not reinstall repeatedly). No Drive mount or local project files are needed.

The default model is `gpt-4o-mini`. API billing/quota and model access are required; a ChatGPT subscription does not supply API quota. Keys are never printed. Package conflicts can occur if multiple framework notebooks share one runtime—start a fresh session when switching frameworks.


In [ ]:
import sys
if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("CrewAI requires Python 3.10–3.13. Choose a compatible Colab runtime.")
%pip -q install "crewai==1.15.27" "python-dotenv==1.2.4"


## 2. Credentials and model
The original model name is retained. API access to this name is not verified. If your endpoint does not support it, set MODEL_NAME to an accessible model consistently across the comparison notebooks. OPENAI_BASE_URL can point to an existing compatible endpoint.

In [ ]:
MODEL_NAME = "gpt-4o-mini"  # Public OpenAI API model.

In [ ]:
import os
import getpass
from dotenv import load_dotenv, find_dotenv


def read_openai_key():
    # In Colab, always use this notebook's enabled secret.
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    if userdata is not None:
        try:
            key = userdata.get("OPENAI_API_KEY")
        except Exception:
            raise RuntimeError(
                "In Colab Secrets, add OPENAI_API_KEY and enable Notebook access, "
                "then rerun this cell."
            ) from None
        if not isinstance(key, str) or not key.strip():
            raise RuntimeError("The OPENAI_API_KEY Colab secret is empty.")
        return key.strip()

    # Local Python fallback; never overwrite an existing environment key.
    try:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv(usecwd=True), override=False)
    except ImportError:
        pass
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not key:
        from getpass import getpass
        key = getpass("Enter OPENAI_API_KEY (hidden): ").strip()
    if not key:
        raise ValueError("OPENAI_API_KEY cannot be empty.")
    return key

os.environ["OPENAI_API_KEY"] = read_openai_key()
print("OpenAI key loaded securely; authentication has not yet been tested.")

## 3. Direct model call
The ask helper sends a prompt directly to CrewAI’s LLM and returns plain text. The Flow below handles step order and shared state.

Fresh instances prevent conversation carryover between calls. Delegation, memory, tracing and caching stay off. max_iter limits the internal agent loop, but a framework retry can still mean more than five HTTP requests. Five task invocations are the workflow contract.


In [ ]:
os.environ["CREWAI_TELEMETRY_DISABLED"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"
from crewai import LLM
from crewai.flow.flow import Flow, start, listen
SAMPLE_TICKET = "My account is locked and I have an important meeting in 10 minutes."

async def ask(prompt: str) -> str:
    llm = LLM(model="openai/" + MODEL_NAME,
              base_url=os.getenv("OPENAI_BASE_URL") or "https://api.openai.com/v1")
    return (await llm.acall(prompt)).strip()


## 4. Native Flow state
The business functions below read and write the Flow state passed to them. Thin decorated methods define only the execution order.

Flow.state starts with its UUID and the input message. Cleaning stores ticket, then later methods create analysis, decision and reply. This mutable state has explicit assignments, without LangGraph reducers. We return only the four original business keys. A fresh Flow isolates each ticket. No persistence or checkpoint recovery is configured. CrewAI 1.15.27 creates a Flow memory facility by default, but this pipeline never calls remember or recall.

The summary, category and urgency tasks run sequentially, matching the LangGraph and Microsoft examples. The LCEL example uses RunnableParallel for those calls. Async syntax alone does not make sequential awaits overlap.


In [ ]:
async def clean_step(state):
    ticket = state['message'].strip()
    if not ticket:
        raise ValueError('Ticket cannot be empty.')
    state['ticket'] = ticket

async def analyse_ticket(state):
    ticket = state['ticket']
    summary = (await ask(f'Summarize this support ticket in one short sentence:\n{ticket}')).strip()
    category = (await ask(f'Return only one category: billing, technical, or account.\nTicket: {ticket}')).strip()
    urgency = (await ask(f'Return only one urgency: low, medium, or high.\nTicket: {ticket}')).strip()
    analysis: dict = {'summary': summary, 'category': category, 'urgency': urgency}
    state['analysis'] = analysis

async def add_decision(state):
    ticket = state['ticket']
    analysis = state['analysis']
    prompt = f"""
    Review this ticket and the initial analysis:

    Ticket:
    {ticket}

    Analysis:
    {analysis}

    Give the final decision:
    1. Should this ticket be escalated to a human? (Yes/No)
    2. Brief reason why.
    """
    decision = (await ask(prompt)).strip()
    state['decision'] = decision

async def add_reply(state):
    ticket = state['ticket']
    decision = state['decision']
    prompt = f"""
    You are a customer-support assistant.

    Customer message:
    {ticket}

    Decision:
    {decision}

    Write a short and helpful reply to the customer.
    Do not mention prompts, AI, or internal system details.
    """
    reply = (await ask(prompt)).strip()
    state['reply'] = reply
    result = {'ticket': state['ticket'], 'analysis': state['analysis'], 'decision': state['decision'], 'reply': state['reply']}
    return result


# Only the workflow connections live here.
class TicketFlow(Flow):
    @start()
    async def clean(self):
        await clean_step(self.state)

    @listen(clean)
    async def analyse(self):
        await analyse_ticket(self.state)

    @listen(analyse)
    async def decide(self):
        await add_decision(self.state)

    @listen(decide)
    async def reply(self):
        return await add_reply(self.state)


## 5. One caller function
Use top-level await in notebooks. The return shape matches all companion examples. The decision recommends escalation. It does not contact anyone or unlock an account.

In [ ]:
async def solve_ticket(message: str) -> dict:
    result = await TicketFlow().kickoff_async(inputs={"message": message})

    print("\n========== TICKET RESULT (CrewAI) ==========")
    print("Message  :", result["ticket"])
    print("\nAnalysis :")
    for key, val in result["analysis"].items():
        print(f"  - {key}: {val}")
    print("\nDecision :\n", result["decision"])
    print("\nReply to customer:\n", result["reply"])
    print("===============================================================")
    return result

In [ ]:
result = await solve_ticket(SAMPLE_TICKET)

## 6. Teaching comparison

| Concept | LCEL | LangGraph | Microsoft | CrewAI here |
|---|---|---|---|---|
| Order | Runnable composition | Nodes and edges | Executors and edges | Flow start/listen |
| Work unit | Runnable | Node | Executor | Flow method calling a business function |
| State | assign preserves fields | Partial updates | get_state/set_state | Mutable Flow.state |
| Analysis scheduling | RunnableParallel | Sequential | Sequential | Sequential |
| Answer | Parsed text | Node values | AgentResponse.text | LLM response text |
| Tools | None | None | None | None |

Plain-text category, urgency and decision outputs have no schema enforcement. Model wording may vary. For the sample, account/high/Yes are plausible, not guaranteed. Try whitespace-only input and verify it fails before model tasks. Then run two distinct messages and inspect state isolation.

[Flow state](https://docs.crewai.com/en/concepts/flows), [LLMs](https://docs.crewai.com/en/concepts/llms).

Validated locally with pinned CrewAI 1.15.27 and a deterministic BaseLLM in the separate developer script. Real API authentication, support for the inherited model name, hosted Colab installation and model quality remain untested. Saved outputs are cleared.
